# 2.2 Apriori implementation
In this exercise, you will implement your own version of Apriori and use it on the COCO dataset to
extract frequent itemsets.

# COCO Dataset
COCO Dataset is a large-scale object detection, segmentation, and captioning dataset. It offers a large number of images (from various contexts) with annotations (i.e. structured information on the contents of the image). These annotations, in particular, regard the contents of the image and, in particular, the objects contained within.

You can download a filtered and preprocessed version of COCO (which we will refer to as “modified”) from the following URL: https://raw.githubusercontent.com/dbdmg/data-science-lab/master/datasets/modified_coco.json  

This dataset is a JSON file. You can open it using the already introduced json module. The file contains a list of images and, for each image, the annotation key contains all the annotations available. The following are the annotations for one such image.

```json
{
"file_name": "000000465265.png",
"image_id": 465265,
"annotations": [
"person",
"person",
"person",
"fire hydrant",
"handbag",
"chair",
"cell phone"
]
}
```

This means that the image contains 3 people, a fire hydrant, a handbag, a chair and a cell phone.

Note that, while this entire exercise is optional, we recommend you try to solve it anyway. You may argue that there are libraries already implementing these and other algorithms. Despite that, we believe it is important, for a data scientist, to know the underlying theory as well as some implementation details. You can learn the former on textbooks, but you will only be faced with the latter when actually working on the implementation.

# BAH

(a) You can find a thorough explanation of the Apriori algorithm on the course slides on association rules (slides 21-35). With these, implement your own version of Apriori. You can use the below toy dataset from the example in slides 21-35 for initial troubleshooting and testing.

a,b  
b,c,d  
a,c,d,e  
a,d,e  
a,b,c  
a,b,c,d  
b,c  
a,b,c  
a,b,d  
b,c,e  

When run with minsup > 1 (or 0.1 in relative terms), the expected itemset (with their minsups) are:

a -> 7  
b -> 8  
c -> 7  
d -> 5  
e -> 3  

a,b -> 5  
a,c -> 4  
a,d -> 4  
a,e -> 2  
b,c -> 6  
b,d -> 3  
c,d -> 3  
c,e -> 2  
d,e -> 2  

a,b,c -> 3  
a,b,d -> 2  
a,c,d -> 2  
a,d,e -> 2  
b,c,d -> 2  

In [ ]:
import pandas as pd

In [ ]:
# import json file
df = pd.read_json('../../Dataset/LAB7/modified_coco.json')
display(df)

In [ ]:
toy_sample = [
    ['a','b'],
    ['b','c','d'],
    ['a','c','d','e'],
    ['a','d','e'],
    ['a','b','c'],
    ['a','b','c','d'],
    ['b','c'],
    ['a','b','c'],
    ['a','b','d'],
    ['b','c','e']
    ]
# list of lists

minsup = 1

#### I'll manually implement the algorithm and then I'll build a clean function

In [ ]:
# get unique items --> generate 1-itemset candidates C1
C1 = []
for itemsets in toy_sample:
    for itemset in itemsets:
        if sorted(itemset) not in C1:
            C1.append(sorted(itemset))

print(C1)

# # or flatten the list and then do a set
# all_itemset = []
# for itemsets in toy_sample:
#     for itemset in itemsets:
#         all_itemset.append(itemset)
# C1 = set(all_itemset)
# # print(C1)
# print(sorted(C1))   # sort turns it back to a list

# # or
# unique_items = set()
# for transaction in toy_sample:
#     for item in transaction:
#         unique_items.add(item)

# C1 = sorted(unique_items)
# print(C1)


In [ ]:
# count support for each element in C1 --> if support > minsup --> add the itemset to the frequent 1-itemsets F1
F1 = []
forbidden_itemsets=[]

for candidates in C1:                   # ['a']
    count = 0
    for itemsets in toy_sample:         # ['a','b']
        found = True
        for c in candidates:            # 'a'
            if c in itemsets:
                continue
            else:
                found = False
                break
    
        if found:
            count += 1
    
    if count > minsup:
        F1.append(candidates)
    else:
        forbidden_itemsets.append(candidates)

print(F1)

In [ ]:
# from F1 generate C2
C2 = []
n = len(F1)
for fixed_idx in range(n):
    for varying_idx in range(fixed_idx+1, n):
        fixed_itemset = set(F1[fixed_idx])
        varying_itemset = set(F1[varying_idx])

        candidate = fixed_itemset | varying_itemset         # union as sets (so ['a','b'] + ['a','c'] -> {'a','b','c'})

        C2.append(sorted(candidate))

print(C2)

In [ ]:
# compute support for each element in C2 --> if support < minsup = prune --> otherwise ass it to F2

forbidden_itemsets = []
F2 = []

for candidates in C2:                    # ['a', 'b']
    count = 0
    # print(f"Candidates: {candidates}")
    for itemsets in toy_sample:         # ['b','c','d']
        found = True
        for c in candidates:
            # print(f"searching candidate: {c} from {candidates}")
            # print(f"inside itemset from ds: {itemsets}")
            # print()
            if c in itemsets:
                # we gucci
                # print(f"Found {c} in {itemsets}")
                # print()
                continue
            else:
                # print(f"DID NOT found {c} in {itemsets}")
                # print('skip it')
                # print()
                found = False
                break
        if found:
            # print(f"Found all itemsets of {candidates} in {itemsets}")
            # print()
            count += 1
    if count > minsup:
        # print(f"Candidate {candidates} found {count}")
        # print()
        F2.append(candidates)
    else:
        forbidden_itemsets.append(candidates)

print(forbidden_itemsets)
print(F2)


In [ ]:
# from F2 generate C3
C3 = []
for fixed_idx in range(len(F2)):
    for varying_idx in range(fixed_idx+1, len(F2)):
        fixed_itemset = set(F2[fixed_idx])
        varying_itemset = set(F2[varying_idx])

        candidate = fixed_itemset | varying_itemset

        C3.append(sorted(candidate))     # ['a', 'b'], ['a', 'c']

print(C3)

#### mmhhh, as you can see it aslo generates the candidates not of length 3, as it generates all possible candidates
---> add a check to ensure the correct lenght

#### It also has duplicates, remove them
--> add a checl to ensure the candidate isn't already in the list

In [ ]:
# from F2 generate C3
C3 = []
for fixed_idx in range(len(F2)):
    for varying_idx in range(fixed_idx+1, len(F2)):
        fixed_itemset = set(F2[fixed_idx])
        varying_itemset = set(F2[varying_idx])

        candidate = fixed_itemset | varying_itemset
        
        # add this
        if len(candidate) == 3 and sorted(candidate) not in C3:
            C3.append(sorted(candidate))     # ['a', 'b'], ['a', 'c']

print(C3)

In [ ]:
 # from C3 compute F3
F3 = []
forbidden_itemsets = []

for candidates in C3:
    count = 0
    for itemsets in toy_sample:
        found = True
        for c in candidates:
            if c in itemsets:
                continue
            else:
                found = False
                break
        if found:
            count += 1

    if count > minsup:
        F3.append(candidates)
    else:
        forbidden_itemsets.append(candidates)

print(F3)
print()
print(forbidden_itemsets)



# Nice
Now let's make it clean

In [ ]:
def initialization(DS, verbose=False):
    C1 = []
    for itemsets in DS:
        for itemset in itemsets:
            if sorted(itemset) not in C1:
                C1.append(sorted(itemset))
    if verbose:
        print(C1)
    return C1


def compute_frequent(DS, Ck, minsup, verbose = False):
    Fk = []
    forbidden_itemsets = []

    for candidates in Ck:
        count = 0
        for itemsets in DS:
            found = True
            for c in candidates:
                if c in itemsets:
                    continue
                else:
                    found = False
                    break
            if found:
                count += 1

        if count > minsup:
            Fk.append(candidates)
        else:
            forbidden_itemsets.append(candidates)
    
    if verbose:
        print(Fk)

    return Fk


def generate_candidates(Fk_1, k, verbose = False):
    Ck = []
    for fixed_idx in range(len(Fk_1)):
        for varying_idx in range(fixed_idx+1, len(Fk_1)):
            fixed_itemset = set(Fk_1[fixed_idx])
            varying_itemset = set(Fk_1[varying_idx])

            candidate = fixed_itemset | varying_itemset
            
            if len(candidate) == k and sorted(candidate) not in Ck:
                Ck.append(sorted(candidate))
    if verbose:
        print(Ck)
    return Ck


def home_made_apriori(DS, minsup, verbose = False):
    # initialization --> get unique itemset to start generating candidates + generate candidates C1
    # compute frequent itemset and prune infrequent ones
    # generate following candidates
    # compute following frequent itemset and prune infrequent ones
    # continue untill there aren't any candidates to generate

    k = 1
    C1 = initialization(DS, verbose)
    frequent_itemsets = compute_frequent(DS, C1, minsup, verbose)

    print(f'frequent {k}-itemset: {frequent_itemsets}')
    
    while True:
        k+=1
        new_candidates = generate_candidates(frequent_itemsets, k, verbose)
        frequent_itemsets = compute_frequent(DS, new_candidates, minsup, verbose)
        if new_candidates == []:
            break
        print(f'frequent {k}-itemset: {frequent_itemsets}')


if __name__ == '__main__':
    DS = [
    ['a','b'],
    ['b','c','d'],
    ['a','c','d','e'],
    ['a','d','e'],
    ['a','b','c'],
    ['a','b','c','d'],
    ['b','c'],
    ['a','b','c'],
    ['a','b','d'],
    ['b','c','e']
    ]
    minsup = 1

    home_made_apriori(DS, minsup)

---

#### (b) Once you have implemented a working version of Apriori, you can load the modified COCO dataset from Subsection 1.1.2 into memory.
> From this, you should transform the dataset into a version compatible with the expected input of your Apriori implementation.

In [ ]:
display(df)

df['image_id'].value_counts()
# each image is unique, THANK GOD
# I guess I need to save the annotation for each image into a list and then append it to another list

In [ ]:
DS = df['annotations'].tolist()
print(DS)

#### (c) You can now run your implementation on the modified COCO dataset.
Try using a minsup of 0.02, as well as other values. Are the obtained results meaningful? You can use the COCO dataset “explore” tool to examine any of the image used.

In [ ]:
home_made_apriori(DS, minsup=0.1, verbose=True)

#### Shiiiii
It's broken because of sorted(), it does this:  
"sorted("car") → ['a', 'c', 'r']"  

Which was fine when I had single letters, you can remove it now, like this:

In [ ]:
# without sorted
def initialization(DS, verbose=False):
    C1 = []
    for itemsets in DS:
        for itemset in itemsets:
            if itemset not in C1:
                C1.append([itemset])
    if verbose:
        print('initialization ...')
        print(f'C1: {C1}')
        print()
    return C1


def compute_frequent(DS, Ck, minsup, verbose = False):
    Fk = []
    forbidden_itemsets = []

    for candidates in Ck:
        #print(f'considering candidate: {candidates}')
        count = 0
        for itemsets in DS:
            #print(f'inside the itemset: {itemsets}')
            found = True
            for c in candidates:
                if c in itemsets:
                    #print(f'Found {c} inside {itemsets}')
                    continue
                else:
                    #print(f'DID NOT find {c} inside {itemsets}')
                    found = False
                    break
            if found:
                #print(f"Found {candidates} inside {itemsets}")
                count += 1

        if count > minsup:
            Fk.append(candidates)
        else:
            forbidden_itemsets.append(candidates)
    
    if verbose:
        print('computing frequent ...')
        print(Fk)
        print()

    return Fk


def generate_candidates(Fk_1, k, verbose = False):
    Ck = []
    for fixed_idx in range(len(Fk_1)):
        for varying_idx in range(fixed_idx+1, len(Fk_1)):
            fixed_itemset = set(Fk_1[fixed_idx])
            varying_itemset = set(Fk_1[varying_idx])

            candidate = fixed_itemset | varying_itemset
            
            if len(candidate) == k and candidate not in Ck:
                Ck.append(candidate)
    if verbose:
        print('generating candidates ...')
        print(Ck)
        print()
    return Ck


def home_made_apriori(DS, minsup, verbose = False):
    # initialization --> get unique itemset to start generating candidates + generate candidates C1
    # compute frequent itemset and prune infrequent ones
    # generate following candidates
    # compute following frequent itemset and prune infrequent ones
    # continue untill there aren't any candidates to generate

    k = 1
    C1 = initialization(DS, verbose)
    frequent_itemsets = compute_frequent(DS, C1, minsup, verbose)

    print(f'frequent {k}-itemset: {frequent_itemsets}')
    
    while True:
        k+=1
        new_candidates = generate_candidates(frequent_itemsets, k, verbose)
        frequent_itemsets = compute_frequent(DS, new_candidates, minsup, verbose)
        if new_candidates == []:
            break
        print(f'frequent {k}-itemset: {frequent_itemsets}')


if __name__ == '__main__':
    df = pd.read_json('../../Dataset/LAB7/modified_coco.json')
    DS = df['annotations'].tolist()
    minsup = 1000

    home_made_apriori(DS, minsup, verbose=False)

#### I can now say that using minsup not as a support (so in %) but as an actual number was dumb as fuck
--> fix it

- Also look better, the frequent 1-itemset have duplicates dumbass.

In [ ]:
# without sorted
def initialization(DS, verbose=False):
    C1 = []
    for itemsets in DS:
        for itemset in itemsets:
            if [itemset] not in C1:
                C1.append([itemset])
    if verbose:
        print('initialization ...')
        print(f'C1: {C1}')
        print()
    return C1


def compute_frequent(DS, Ck, minsup, verbose = False):
    Fk = []
    forbidden_itemsets = []

    for candidates in Ck:
        #print(f'considering candidate: {candidates}')
        count = 0
        for itemsets in DS:
            #print(f'inside the itemset: {itemsets}')
            found = True
            for c in candidates:
                if c in itemsets:
                    #print(f'Found {c} inside {itemsets}')
                    continue
                else:
                    #print(f'DID NOT find {c} inside {itemsets}')
                    found = False
                    break
            if found:
                #print(f"Found {candidates} inside {itemsets}")
                count += 1

        # this changed
        support = count / len(DS)
        if support > minsup:
            Fk.append(list(candidates))
        else:
            forbidden_itemsets.append(candidates)
    
    if verbose:
        print('computing frequent ...')
        print(Fk)
        print()

    return Fk


def generate_candidates(Fk_1, k, verbose = False):
    Ck = []
    for fixed_idx in range(len(Fk_1)):
        for varying_idx in range(fixed_idx+1, len(Fk_1)):
            fixed_itemset = set(Fk_1[fixed_idx])
            varying_itemset = set(Fk_1[varying_idx])

            candidate = fixed_itemset | varying_itemset
            
            if len(candidate) == k and candidate not in Ck:
                Ck.append(candidate)
    if verbose:
        print('generating candidates ...')
        print(Ck)
        print()
    return Ck


def home_made_apriori(DS, minsup, verbose = False):
    # initialization --> get unique itemset to start generating candidates + generate candidates C1
    # compute frequent itemset and prune infrequent ones
    # generate following candidates
    # compute following frequent itemset and prune infrequent ones
    # continue untill there aren't any candidates to generate

    k = 1
    C1 = initialization(DS, verbose)
    frequent_itemsets = compute_frequent(DS, C1, minsup, verbose)

    print(f'frequent {k}-itemset: {frequent_itemsets}')
    
    while True:
        k+=1
        new_candidates = generate_candidates(frequent_itemsets, k, verbose)
        frequent_itemsets = compute_frequent(DS, new_candidates, minsup, verbose)
        if new_candidates == []:
            break
        print(f'frequent {k}-itemset: {frequent_itemsets}')


if __name__ == '__main__':
    df = pd.read_json('../../Dataset/LAB7/modified_coco.json')
    DS = df['annotations'].tolist()
    minsup = 0.02

    home_made_apriori(DS, minsup, verbose=False)

In [ ]:
C1 = []
for itemsets in DS:
    print(f'we are considering this itemsets: {itemsets}')
    for itemset in itemsets:
        print(f'Pick {itemset}')
        print(f'Is it already in {C1}?')
        if [itemset] not in C1:
            print('NO, append')
            print()
            C1.append([itemset])
        else:
            print('YEP, skip')
            print()
if True:
    print('initialization ...')
    print(f'C1: {C1}')
    print()

#### Now convert the modified COCO dataset into the format required by Mlxtend’s apriori() and fpgrowth() functions (i.e. the one described in Exercise 3).
Then, run these two algorithms on this dataset. Do the results from these functions match your results?

To run the apriori needs an encoded dataframe:
```python
dataset = [['Milk', 'Onion', 'Nutmeg', 'Kidney Beans', 'Eggs', 'Yogurt'],
           ['Dill', 'Onion', 'Nutmeg', 'Kidney Beans', 'Eggs', 'Yogurt'],
           ['Milk', 'Apple', 'Kidney Beans', 'Eggs'],
           ['Milk', 'Unicorn', 'Corn', 'Kidney Beans', 'Yogurt'],
           ['Corn', 'Onion', 'Onion', 'Kidney Beans', 'Ice cream', 'Eggs']]

te = TransactionEncoder()
te_ary = te.fit(dataset).transform(dataset)
df = pd.DataFrame(te_ary, columns=te.columns_)

Apple	Corn	Dill	Eggs	Ice cream	Kidney Beans	Milk	Nutmeg	Onion	Unicorn	Yogurt
0	False	False	False	True	False	True	True	True	True	False	True
1	False	False	True	True	False	True	False	True	True	False	True
...
```


In [ ]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpgrowth

In [ ]:
OHE = TransactionEncoder()
DS_OHE = OHE.fit_transform(DS)
DF_OHE = pd.DataFrame(DS_OHE, columns=OHE.columns_)
display(DF_OHE)

In [ ]:
frequent_itemsets = apriori(df=DF_OHE, min_support=0.02, use_colnames=True)
display(frequent_itemsets)

# for itemset in frequent_itemsets['itemsets'].values:
#     if len(itemset) == 3:
#         print(itemset)

# or use .map

mask = frequent_itemsets['itemsets'].map(len) == 1
display(frequent_itemsets[mask])

display(frequent_itemsets.sort_values(by='support', ascending=False))

In [ ]:
frequent_itemsets = fpgrowth(DF_OHE, min_support=0.02, use_colnames=True)
display(frequent_itemsets)

display(frequent_itemsets.sort_values(by='support', ascending=False))

---

- For regression try these out

# RidgeCV
# LassoCV